<a href="https://www.kaggle.com/code/emrebaranarca/cnn-digit-recognizer?scriptVersionId=300819845" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import matplotlib.pyplot as plt

train_df = pd.read_csv('/kaggle/input/digit-recognizer/train.csv')
test_df = pd.read_csv('/kaggle/input/digit-recognizer/test.csv')

print(f"Train: {train_df.shape}")
print(f"Test: {test_df.shape}")
print(np.unique(train_df.iloc[0, 1:].values)[:20])  # ilk 20 farklı değer


n = 12
idx = np.random.choice(len(train_df), n, replace=False)

plt.figure(figsize=(10, 7))
for k, i in enumerate(idx):
    etiket = train_df.iloc[i, 0]
    resim = train_df.iloc[i, 1:].values.reshape(28, 28)

    plt.subplot(3, 4, k+1)
    plt.imshow(resim, cmap="gray")
    plt.title(f"{etiket}")
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
labels = train_df['label'].to_numpy()
train_images=train_df.drop('label',axis=1).to_numpy().astype(np.float32)/255.0

plt.imshow(train_images[0].reshape(28,28), cmap="gray")
plt.title(labels[0])
plt.axis("off")
plt.show()
print(train_images.shape)


In [ ]:
test_images=test_df.to_numpy().astype(np.float32) / 255.0
plt.imshow(test_images[0].reshape(28,28), cmap="gray")
plt.title(labels[0])
plt.axis("off")
plt.show()

In [ ]:
print(train_images.shape)
train_images = train_images.reshape(-1, 1, 28, 28)
test_images = test_images.reshape(-1, 1, 28, 28)
print(train_images.shape)
print(test_images.shape)



In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import random

class MNISTDataset(Dataset):
    def __init__(self,images,labels=None,transform=None):
        self.images=torch.tensor(images,dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.long) if labels is not None else None
        self.transform=transform
        

    def __len__(self):
        return len(self.images)

    def __getitem__(self,idx):
        img=self.images[idx]
        if self.transform:
            img=self.transform(img)
        if self.labels is not None:
            return img,self.labels[idx]
        return img


train_transform=transforms.Compose([
    transforms.RandomAffine(
        degrees=15,
        translate=(0.1,0.1),
        scale=(0.9,1.1),
        shear=10
    ),
    transforms.RandomPerspective(distortion_scale=0.2,p=0.5)
])


ds = MNISTDataset(train_images, labels, transform=train_transform)
x, y = ds[0]
print(x.shape, x.dtype, y)

plt.imshow(x.squeeze(0), cmap="gray")
plt.title(int(y))
plt.axis("off")
plt.show()

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_ch)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_ch != out_ch:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_ch)
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        return F.relu(out)


class ResNetMNIST(nn.Module):
    """ResNet-style CNN for 28x28 grayscale."""
    def __init__(self, num_classes=10):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(32)
        self.layer1 = self._make_layer(32, 64, 2, stride=1)
        self.layer2 = self._make_layer(64, 128, 2, stride=2)
        self.layer3 = self._make_layer(128, 256, 2, stride=2)
        self.dropout = nn.Dropout(0.4)
        self.fc = nn.Linear(256, num_classes)

    def _make_layer(self, in_ch, out_ch, num_blocks, stride):
        layers = [ResidualBlock(in_ch, out_ch, stride)]
        for _ in range(1, num_blocks):
            layers.append(ResidualBlock(out_ch, out_ch))
        return nn.Sequential(*layers)

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = F.adaptive_avg_pool2d(x, 1).view(x.size(0), -1)
        x = self.dropout(x)
        return self.fc(x)


class WideConvNet(nn.Module):
    """Wider VGG-style CNN."""
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            # Block 1
            nn.Conv2d(1, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d(2), nn.Dropout(0.25),
            # Block 2
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.Conv2d(128, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.MaxPool2d(2), nn.Dropout(0.25),
            # Block 3
            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(),
            nn.Conv2d(256, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(),
            nn.MaxPool2d(2, padding=1), nn.Dropout(0.25),
        )
        self.classifier = nn.Sequential(
            nn.Linear(256 * 4 * 4, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)

In [ ]:
def train_one_epoch(model, loader, optimizer, scheduler, device):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for imgs, targets in loader:
        imgs, targets = imgs.to(device), targets.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = F.cross_entropy(outputs, targets, label_smoothing=0.1)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * imgs.size(0)
        correct += (outputs.argmax(1) == targets).sum().item()
        total += imgs.size(0)
    if scheduler:
        scheduler.step()
    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    for imgs, targets in loader:
        imgs, targets = imgs.to(device), targets.to(device)
        outputs = model(imgs)
        loss = F.cross_entropy(outputs, targets)
        total_loss += loss.item() * imgs.size(0)
        correct += (outputs.argmax(1) == targets).sum().item()
        total += imgs.size(0)
    return total_loss / total, correct / total

In [ ]:
# ============================================================
# 5. K-Fold Training with Ensemble
# 5-fold CV x 2 architectures = 10 models
# ============================================================
from sklearn.model_selection import StratifiedKFold
import torch.optim as optim
import torch.nn.functional as F
import warnings
warnings.filterwarnings('ignore')


# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')



N_FOLDS = 5
EPOCHS = 30
BATCH_SIZE = 128
LR = 1e-3

models_all = []
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

for model_class, model_name in [(ResNetMNIST, 'ResNet'), (WideConvNet, 'WideConv')]:
    print(f'\n{"="*60}')
    print(f'Training {model_name}')
    print(f'{"="*60}')

    for fold, (train_idx, val_idx) in enumerate(skf.split(train_images, labels)):
        print(f'\nFold {fold+1}/{N_FOLDS}')

        train_ds = MNISTDataset(train_images[train_idx], labels[train_idx], transform=train_transform)
        val_ds = MNISTDataset(train_images[val_idx], labels[val_idx], transform=None)

        train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
        val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE*2, shuffle=False, num_workers=2, pin_memory=True)

        model = model_class().to(device)
        optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
        scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

        best_val_acc = 0
        best_state = None
        patience_counter = 0

        for epoch in range(EPOCHS):
            train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, scheduler, device)
            val_loss, val_acc = evaluate(model, val_loader, device)

            if val_acc > best_val_acc:
                best_val_acc = val_acc
                best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
                patience_counter = 0
            else:
                patience_counter += 1

            if (epoch + 1) % 5 == 0 or epoch == 0:
                print(f'  Epoch {epoch+1:02d} | Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} Acc: {val_acc:.4f}')

            if patience_counter >= 10:
                print(f'  Early stopping at epoch {epoch+1}')
                break

        model.load_state_dict(best_state)
        model.to(device)
        model.eval()
        models_all.append(model)
        print(f'  Best Val Acc: {best_val_acc:.5f}')

print(f'\nTotal models in ensemble: {len(models_all)}')


In [ ]:
tta_transforms = [
    None,
    transforms.RandomAffine(degrees=5, translate=(0.05, 0.05)),
    transforms.RandomAffine(degrees=(-5, -5), translate=(0.05, 0.05)),
    transforms.RandomAffine(degrees=0, scale=(0.95, 1.05)),
]


@torch.no_grad()
def predict_ensemble(models, test_images, device, tta_transforms, batch_size=256):
    all_probs = torch.zeros(len(test_images), 10)

    for model in models:
        model.eval()
        for tta_tf in tta_transforms:
            ds = MNISTDataset(test_images, labels=None, transform=tta_tf)
            loader = DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=2)
            probs_list = []
            for imgs in loader:
                imgs = imgs.to(device)
                logits = model(imgs)
                probs_list.append(F.softmax(logits, dim=1).cpu())
            all_probs += torch.cat(probs_list, dim=0)

    predictions = all_probs.argmax(dim=1).numpy()
    return predictions


predictions = predict_ensemble(models_all, test_images, device, tta_transforms)
print(f'Predictions shape: {predictions.shape}')
print(f'Prediction distribution: {np.bincount(predictions)}')

In [ ]:
submission = pd.DataFrame({
    'ImageId': range(1, len(predictions) + 1),
    'Label': predictions
})
submission.to_csv('submission.csv', index=False)
print(f'Submission saved: {submission.shape}')
print(submission.head(10))